# Task 1 — Scraping and Preprocessing Google Play Reviews

**Project:** Customer Experience Analytics for Fintech Apps

This notebook follows the same teaching style as the demo notebook, but it is adapted for the three target banks in the challenge:
- Commercial Bank of Ethiopia (CBE)
- Bank of Abyssinia (BOA)
- Dashen Bank

The goal of Task 1 is to collect raw Google Play Store reviews, clean them, and save a structured CSV that is ready for sentiment and thematic analysis.

**Final output columns:** `review`, `rating`, `date`, `bank`, `source`

---
# Setup
The notebook also uses standard library modules for cleaning and file handling.

In [30]:
# Core libraries
import os
import re
from datetime import datetime

import numpy as np
import pandas as pd

# Google Play Store scraper
from google_play_scraper import app, reviews, Sort

pd.set_option('display.max_colwidth', 120)

print("Libraries loaded successfully!")

Libraries loaded successfully!


---
## Bank Configuration

We define one place for all three apps so the scraping code stays reusable and easy to maintain.

The `source_url` field is stored as the Google Play URL-like identifier with the app id, which makes the dataset easier to trace back to the original app listing.

In [31]:
BANKS = [
    {
        'bank': 'Commercial Bank of Ethiopia',
        'app_name': 'CBE Mobile Banking',
        'app_id': 'com.combanketh.mobilebanking',
        'source': 'google_play',
        'source_url': 'https://play.google.com/store/apps/details?id=com.combanketh.mobilebanking',
    },
    {
        'bank': 'Bank of Abyssinia',
        'app_name': 'BOA Mobile Banking',
        'app_id': 'com.boa.boaMobileBanking',
        'source': 'google_play',
        'source_url': 'https://play.google.com/store/apps/details?id=com.boa.boaMobileBanking',
    },
    {
        'bank': 'Dashen Bank',
        'app_name': 'Dashen SuperApp',
        'app_id': 'com.dashen.dashensuperapp',
        'source': 'google_play',
        'source_url': 'https://play.google.com/store/apps/details?id=com.dashen.dashensuperapp',
    }
]

for bank in BANKS:
    print(f"{bank['bank']}: {bank['app_id']}")

Commercial Bank of Ethiopia: com.combanketh.mobilebanking
Bank of Abyssinia: com.boa.boaMobileBanking
Dashen Bank: com.dashen.dashensuperapp


---
# App Metadata

Before scraping reviews, we first inspect each app listing. This helps confirm that the app id is valid and gives a quick summary of app quality signals like rating and total reviews.

In [32]:
app_rows = []

for bank in BANKS:
    try:
        app_info = app(
            bank['app_id'],
            lang='en',
            country='et'
        )

        app_rows.append({
            'bank': bank['bank'],
            'app_name': bank['app_name'],
            'title': app_info.get('title'),
            'score': app_info.get('score'),
            'ratings': app_info.get('ratings'),
            'reviews': app_info.get('reviews'),
            'installs': app_info.get('installs')
        })
    except Exception as exc:
        app_rows.append({
            'bank': bank['bank'],
            'app_name': bank['app_name'],
            'title': None,
            'score': np.nan,
            'ratings': np.nan,
            'reviews': np.nan,
            'installs': f'Error: {exc}'
        })

df_apps = pd.DataFrame(app_rows)
df_apps

,bank,app_name,title,score,ratings,reviews,installs
0,Commercial Bank of Ethiopia,CBE Mobile Banking,Commercial Bank of Ethiopia,4.287285,48500,9323,"5,000,000+"
1,Bank of Abyssinia,BOA Mobile Banking,BoA Mobile,4.393908,9265,1463,"1,000,000+"
2,Dashen Bank,Dashen SuperApp,Dashen Bank,4.246180,5675,1026,"1,000,000+"


---
# Scraping Reviews

We scrape up to 500 reviews per bank so that we can comfortably satisfy the minimum target of 400 reviews per bank, if the app listing returns enough data.

Each scraped record will keep the review text, star rating, posting date, bank name, source, and review id for duplicate handling.

In [33]:
def scrape_reviews_for_bank(bank_config, count=500):
    """Scrape reviews for one bank and return a list of cleaned raw rows."""
    print(f"Scraping reviews for {bank_config['bank']}...")

    result, continuation_token = reviews(
        bank_config['app_id'],
        lang='en',
        country='et',
        sort=Sort.NEWEST,
        count=count,
        filter_score_with=None
    )

    rows = []
    for review_item in result:
        rows.append({
            'review_id': review_item.get('reviewId', ''),
            'review': review_item.get('content', ''),
            'rating': review_item.get('score', np.nan),
            'date': review_item.get('at', None),
            'bank': bank_config['bank'],
            'app_name': bank_config['app_name'],
            'source': bank_config['source'],
            'source_url': bank_config['source_url']
        })

    print(f"Collected {len(rows)} raw reviews for {bank_config['bank']}")
    return rows

raw_rows = []
for bank in BANKS:
    raw_rows.extend(scrape_reviews_for_bank(bank, count=500))

df_raw = pd.DataFrame(raw_rows)
print(f"Combined raw dataset shape: {df_raw.shape}")
df_raw.head()

Scraping reviews for Commercial Bank of Ethiopia...
Collected 500 raw reviews for Commercial Bank of Ethiopia
Scraping reviews for Bank of Abyssinia...
Collected 500 raw reviews for Bank of Abyssinia
Scraping reviews for Dashen Bank...
Collected 500 raw reviews for Dashen Bank
Combined raw dataset shape: (1500, 8)


,review_id,review,rating,date,bank,app_name,source,source_url
0,2ae8efd1-7515-4a39-a2dd-7f3e84ee190a,ok,5,2026-05-16 21:47:39,Commercial Bank of Ethiopia,CBE Mobile Banking,google_play,https://play.google.com/store/apps/details?id=com.combanketh.mobilebanking
1,f11ba9ef-c1a1-4006-9ead-afb585624f63,Good,5,2026-05-16 19:03:11,Commercial Bank of Ethiopia,CBE Mobile Banking,google_play,https://play.google.com/store/apps/details?id=com.combanketh.mobilebanking
2,5c08f975-fcca-4b2d-8044-25490b83e988,🤙🏼🤙🏼,5,2026-05-16 15:50:50,Commercial Bank of Ethiopia,CBE Mobile Banking,google_play,https://play.google.com/store/apps/details?id=com.combanketh.mobilebanking
3,c1e25b5d-7e79-4b60-8aa5-bed69a904f62,worst,1,2026-05-16 12:15:55,Commercial Bank of Ethiopia,CBE Mobile Banking,google_play,https://play.google.com/store/apps/details?id=com.combanketh.mobilebanking
4,eb3cc438-1c10-4e72-8851-3efff6a04135,this app very full,5,2026-05-16 09:17:00,Commercial Bank of Ethiopia,CBE Mobile Banking,google_play,https://play.google.com/store/apps/details?id=com.combanketh.mobilebanking


---
# Explore the Raw Data

Before cleaning, we inspect the raw output to understand what the scraper returned and whether the dataset is ready for preprocessing.

In [34]:
print(f"Total raw reviews collected: {len(df_raw)}")
print("Rows per bank:")
print(df_raw['bank'].value_counts())

print("Column types:")
print(df_raw.dtypes)

print("Sample raw reviews:")
df_raw[['bank', 'rating', 'date', 'review']].head(10)

Total raw reviews collected: 1500
Rows per bank:
bank
Commercial Bank of Ethiopia    500
Bank of Abyssinia              500
Dashen Bank                    500
Name: count, dtype: int64
Column types:
review_id                str
review                   str
rating                 int64
date          datetime64[us]
bank                     str
app_name                 str
source                   str
source_url               str
dtype: object
Sample raw reviews:


,bank,rating,date,review
0,Commercial Bank of Ethiopia,5,2026-05-16 21:47:39,ok
1,Commercial Bank of Ethiopia,5,2026-05-16 19:03:11,Good
2,Commercial Bank of Ethiopia,5,2026-05-16 15:50:50,🤙🏼🤙🏼
3,Commercial Bank of Ethiopia,1,2026-05-16 12:15:55,worst
4,Commercial Bank of Ethiopia,5,2026-05-16 09:17:00,this app very full
5,Commercial Bank of Ethiopia,4,2026-05-16 07:18:33,good apps
6,Commercial Bank of Ethiopia,5,2026-05-16 03:43:47,ok
7,Commercial Bank of Ethiopia,1,2026-05-15 23:20:32,"this update got crazy i don't know what's going on this app it's mal functional and loading........... like 2G ,Haha"
8,Commercial Bank of Ethiopia,5,2026-05-15 20:11:22,thanks for you 😘
9,Commercial Bank of Ethiopia,4,2026-05-15 19:53:26,it's okay


In [35]:
print("Missing values by column:")
missing_counts = df_raw.isnull().sum()
missing_pct = (missing_counts / len(df_raw) * 100).round(2)

for column in df_raw.columns:
    print(f"{column:<15}: {missing_counts[column]} missing ({missing_pct[column]}%)")

print("\nDuplicate review ids:", df_raw.duplicated(subset=['review_id']).sum())
print("Duplicate review text:", df_raw.duplicated(subset=['review']).sum())

Missing values by column:
review_id      : 0 missing (0.0%)
review         : 0 missing (0.0%)
rating         : 0 missing (0.0%)
date           : 0 missing (0.0%)
bank           : 0 missing (0.0%)
app_name       : 0 missing (0.0%)
source         : 0 missing (0.0%)
source_url     : 0 missing (0.0%)

Duplicate review ids: 0
Duplicate review text: 366


---
# Cleaning and Preprocessing

The cleaning rules are simple and business-focused:
- Drop rows missing review text or rating
- Remove duplicate reviews
- Normalize dates to `YYYY-MM-DD`
- Strip extra whitespace from review text
- Keep only valid star ratings from 1 to 5
- Save only the required final columns

In [36]:
def clean_review_text(text):
    """Collapse repeated whitespace and trim the review text."""
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df_clean_working = df_raw.copy()
before_rows = len(df_clean_working)

# Drop missing critical fields
df_clean_working = df_clean_working.dropna(subset=['review', 'rating'])
after_missing_drop = len(df_clean_working)

# Remove duplicate review ids and duplicate text
df_clean_working = df_clean_working.drop_duplicates(subset=['review_id'], keep='first')
df_clean_working = df_clean_working.drop_duplicates(subset=['review'], keep='first')
after_duplicates_drop = len(df_clean_working)

# Clean text and remove empty strings that may remain
df_clean_working['review'] = df_clean_working['review'].apply(clean_review_text)
df_clean_working = df_clean_working[df_clean_working['review'].str.len() > 0]
after_text_clean = len(df_clean_working)

# Normalize dates
df_clean_working['date'] = pd.to_datetime(df_clean_working['date'], errors='coerce').dt.strftime('%Y-%m-%d')
df_clean_working = df_clean_working.dropna(subset=['date'])

# Keep only valid ratings
df_clean_working = df_clean_working[df_clean_working['rating'].between(1, 5)]
df_clean_working['rating'] = df_clean_working['rating'].astype(int)

# Final output columns only
df_clean = df_clean_working[['review', 'rating', 'date', 'bank', 'source']].copy()

print(f"Rows before cleaning: {before_rows}")
print(f"After dropping missing critical fields: {after_missing_drop}")
print(f"After removing duplicates: {after_duplicates_drop}")
print(f"After text cleaning: {after_text_clean}")
print(f"Final cleaned rows: {len(df_clean)}")
df_clean.head()

Rows before cleaning: 1500
After dropping missing critical fields: 1500
After removing duplicates: 1134
After text cleaning: 1134
Final cleaned rows: 1134


,review,rating,date,bank,source
0,ok,5,2026-05-16,Commercial Bank of Ethiopia,google_play
1,Good,5,2026-05-16,Commercial Bank of Ethiopia,google_play
2,🤙🏼🤙🏼,5,2026-05-16,Commercial Bank of Ethiopia,google_play
3,worst,1,2026-05-16,Commercial Bank of Ethiopia,google_play
4,this app very full,5,2026-05-16,Commercial Bank of Ethiopia,google_play


### Per-bank counts after cleaning

This helps confirm whether each bank still has enough reviews for analysis.

In [37]:
bank_counts_raw = df_raw['bank'].value_counts().rename('raw_count')
bank_counts_clean = df_clean['bank'].value_counts().rename('clean_count')
bank_summary = pd.concat([bank_counts_raw, bank_counts_clean], axis=1).fillna(0).astype(int)
bank_summary['retention_rate_%'] = (bank_summary['clean_count'] / bank_summary['raw_count'] * 100).round(1)
bank_summary

,raw_count,clean_count,retention_rate_%
bank,,,
Commercial Bank of Ethiopia,500,374,74.8
Bank of Abyssinia,500,378,75.6
Dashen Bank,500,382,76.4


---
# Save the Clean Dataset

The final CSV should be treated as a generated artifact, so it should be excluded from GitHub with `.gitignore`. The file below is the cleaned, analysis-ready output for the rest of the project.

In [38]:
output_dir = os.path.join('data', 'processed')
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'ethiopian_bank_reviews_clean.csv')
df_clean.to_csv(output_path, index=False)

print(f"Saved cleaned dataset to: {output_path}")
print(f"Final dataset shape: {df_clean.shape}")
print("Final columns:", list(df_clean.columns))

Saved cleaned dataset to: data\processed\ethiopian_bank_reviews_clean.csv
Final dataset shape: (1134, 5)
Final columns: ['review', 'rating', 'date', 'bank', 'source']


---
### Reproducible scraping (script)

A standalone script `scripts/src/scrape_preprocess.py` has been added to the repository.
Use the script to re-run scraping and preprocessing outside the notebook, or to reproduce the cleaned CSV when needed. The notebook continues to load `notebooks/data/processed/ethiopian_bank_reviews_clean.csv` by default.

In [39]:
# Run the standalone scraper (from repository root).
# Use --force to overwrite an existing cleaned CSV.
!python scripts/src/scrape_preprocess.py --output notebooks/data/processed/ethiopian_bank_reviews_clean.csv --count 500 --force

# Reload the produced cleaned CSV into the notebook environment using multiple candidate paths
from pathlib import Path
import pandas as pd
candidates = [
    Path('notebooks/data/processed/ethiopian_bank_reviews_clean.csv'),
    Path('../notebooks/data/processed/ethiopian_bank_reviews_clean.csv'),
    Path('data/processed/ethiopian_bank_reviews_clean.csv'),
    Path('..') / 'data' / 'processed' / 'ethiopian_bank_reviews_clean.csv',
]
found = None
for p in candidates:
    if p.exists():
        found = p
        break
if not found:
    raise FileNotFoundError(f'Cleaned CSV not found in any candidate path: {candidates}')
df = pd.read_csv(found)
print(f'Loaded cleaned CSV from: {found}')
df.head()

Loaded cleaned CSV from: ..\notebooks\data\processed\ethiopian_bank_reviews_clean.csv


C:\Users\HP EliteBook\AppData\Local\Programs\Python\Python312\python.exe: can't open file 'c:\\Users\\HP EliteBook\\Desktop\\KAIM\\fintech-review-analytics\\notebooks\\scripts\\src\\scrape_preprocess.py': [Errno 2] No such file or directory


,review,rating,date,bank,source
0,ok,5,2026-05-16,Commercial Bank of Ethiopia,google_play
1,Good,5,2026-05-16,Commercial Bank of Ethiopia,google_play
2,🤙🏼🤙🏼,5,2026-05-16,Commercial Bank of Ethiopia,google_play
3,worst,1,2026-05-16,Commercial Bank of Ethiopia,google_play
4,this app very full,5,2026-05-16,Commercial Bank of Ethiopia,google_play


---
# Preprocessing Report

This short report is the summary you can reuse in README.md or in the interim submission.

In [40]:
print("=" * 60)
print("TASK 1 PREPROCESSING REPORT")
print("=" * 60)

original_total = len(df_raw)
clean_total = len(df_clean)
removed_total = original_total - clean_total
retention_rate = (clean_total / original_total * 100) if original_total else 0

print(f"Raw reviews collected      : {original_total}")
print(f"Clean reviews retained     : {clean_total}")
print(f"Rows removed during cleanup: {removed_total}")
print(f"Retention rate             : {retention_rate:.1f}%")

print("\nPer-bank cleaned counts:")
for bank_name, count in df_clean['bank'].value_counts().items():
    print(f"- {bank_name}: {count}")

print("\nIf any bank returns fewer than 400 reviews, document that limitation in the notebook and README.")
print("=" * 60)

TASK 1 PREPROCESSING REPORT
Raw reviews collected      : 1500
Clean reviews retained     : 1134
Rows removed during cleanup: 366
Retention rate             : 75.6%

Per-bank cleaned counts:
- Dashen Bank: 382
- Bank of Abyssinia: 378
- Commercial Bank of Ethiopia: 374

If any bank returns fewer than 400 reviews, document that limitation in the notebook and README.


---
## Limitations and Notes

- Google Play data availability can vary by app and date range.
- If a bank returns fewer than 400 reviews, the notebook should record the shortfall clearly.
- The CSV output is a generated artifact and should not be committed to GitHub.
- The cleaned dataset is now ready for Task 2 sentiment and thematic analysis.